# Job ETL da silver

Neste notebook, é aplicado o Job ETL. Ele é um processo que extrai, transforma e carrega dados de diferentes fontes para um destino central. No caso desse projeto, a fonte será extraida da camada silver para a gold pelo arquivo Complete_Pokedex-Tratada.csv e o resultado será armazenado em outro csv e utilizado na camada gold.

### Frameworks utilizados

In [14]:
import pandas as pd
import psycopg2
import time

### EXTRACT (Extrair)

Conectando com o banco

In [15]:
def get_db_connection():
    while True:
        try:
            conexao = psycopg2.connect(
                host="localhost",
                port=5432,
                database="pokedex_db",
                user="pokedex_user",
                password="pokedex_password"
            )
            return conexao
        except psycopg2.OperationalError:
            print("O banco não está pronto, aguardando 3 segundos...")
            time.sleep(3)


conexao = get_db_connection()

print("Conexão efetuada com sucesso...")

Conexão efetuada com sucesso...


Extraindo dados do banco

In [16]:
df = pd.read_sql_query("SELECT * FROM slvr.pokemon", conexao)

print("Dados carregados do banco de dados:")
print(df.head())
print(f"Total de registros: {len(df)}")

print("\n Extract concluído!")

Dados carregados do banco de dados:
   pokedex_number   pokemon_name type_1  type_2  height  weight  hit_points  \
0               2        Ivysaur  Grass  Poison     1.0    13.0          60   
1               3  Mega Venusaur  Grass  Poison     2.4   155.5          80   
2               6      Charizard   Fire  Flying     1.7    90.5          78   
3              12     Butterfree    Bug  Flying     1.1    32.0          60   
4              14         Kakuna    Bug  Poison     0.6    10.0          45   

   attack  defense  total_stats  ...  against_ground  against_flying  \
0      62       63          405  ...             1.0             2.0   
1     100      123          625  ...             1.0             2.0   
2      84       78          534  ...             0.0             1.0   
3      45       50          395  ...             0.0             2.0   
4      25       50          205  ...             1.0             2.0   

   against_psychic  against_bug against_rock against_gho

/tmp/ipykernel_430/1870899297.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("SELECT * FROM slvr.pokemon", conexao)


### TRANSFORM (Transformar)


In [17]:
# Apaga colunas
colunas_para_apagar = [
'hit_points',
'base_happiness',
'evolves_from',
'mythical',
'genderless', 
'female_rate', 
'egg_cycles'
]

df_tratado = df.drop(columns=colunas_para_apagar)

print(df_tratado.head())

print("\n Transform concluído!")

   pokedex_number   pokemon_name type_1  type_2  height  weight  attack  \
0               2        Ivysaur  Grass  Poison     1.0    13.0      62   
1               3  Mega Venusaur  Grass  Poison     2.4   155.5     100   
2               6      Charizard   Fire  Flying     1.7    90.5      84   
3              12     Butterfree    Bug  Flying     1.1    32.0      45   
4              14         Kakuna    Bug  Poison     0.6    10.0      25   

   defense  total_stats  capture_rate  ...  against_ground  against_flying  \
0       63          405            45  ...             1.0             2.0   
1      123          625            45  ...             1.0             2.0   
2       78          534            45  ...             0.0             1.0   
3       50          395            45  ...             0.0             2.0   
4       50          205           120  ...             1.0             2.0   

  against_psychic  against_bug  against_rock  against_ghost  against_dragon  \
0

### LOAD (Carregar) 

Criando schema dw e tabelas

In [18]:
cursor = conexao.cursor()

# cria um schema para a silver
cursor.execute("""
CREATE SCHEMA IF NOT EXISTS dw;
""")

# cria Dim_pokmn (POKEMON)
cursor.execute("""
CREATE TABLE IF NOT EXISTS dw.Dim_pkm (
    SRK_pkn SERIAL PRIMARY KEY,
    pkm_nam VARCHAR(50) NOT NULL,
    tp1 VARCHAR(50) NOT NULL,
    tp2 VARCHAR(50),
    hgt DOUBLE PRECISION NOT NULL,
    wgt DOUBLE PRECISION NOT NULL,
    gen INT NOT NULL,
    leg BOOLEAN NOT NULL,
    mga_evl BOOLEAN NOT NULL,
    all_frm BOOLEAN NOT NULL,
    glr_frm BOOLEAN NOT NULL,
    swt_frm BOOLEAN NOT NULL
);
""")

# cria Dim_batlh (BATALHA)
cursor.execute("""
CREATE TABLE IF NOT EXISTS dw.Dim_btl (
    SRK_btl SERIAL PRIMARY KEY,
    atk INT NOT NULL,
    dfs INT NOT NULL,
    cap_rte INT NOT NULL,
    bas_exp INT NOT NULL,
    exp_tpe VARCHAR(50) NOT NULL
);
""")

# cria Dim_efetContr (EFETIVIDADE CONTRA)
cursor.execute("""
CREATE TABLE IF NOT EXISTS dw.Dim_efetContr (
    SRK_eft SERIAL PRIMARY KEY,
    agt_nrm DOUBLE PRECISION NOT NULL,
    agt_fre DOUBLE PRECISION NOT NULL,
    agt_wtr DOUBLE PRECISION NOT NULL,
    agt_elt DOUBLE PRECISION NOT NULL,
    agt_grs DOUBLE PRECISION NOT NULL,
    agt_ice DOUBLE PRECISION NOT NULL,
    agt_fgt DOUBLE PRECISION NOT NULL,
    agt_psn DOUBLE PRECISION NOT NULL,
    agt_gnd DOUBLE PRECISION NOT NULL,
    agt_fly DOUBLE PRECISION NOT NULL,
    agt_psy DOUBLE PRECISION NOT NULL,
    agt_bug DOUBLE PRECISION NOT NULL,
    agt_rck DOUBLE PRECISION NOT NULL,
    agt_gst DOUBLE PRECISION NOT NULL,
    agt_drg DOUBLE PRECISION NOT NULL,
    agt_drk DOUBLE PRECISION NOT NULL,
    agt_stl DOUBLE PRECISION NOT NULL,
    agt_fry DOUBLE PRECISION NOT NULL
);
""")

# cria Fat_pokdx (POKEDEX)
cursor.execute("""
CREATE TABLE IF NOT EXISTS dw.Fat_pokdx (
    SRK_pkx SERIAL PRIMARY KEY,
    SRK_pkn INT NOT NULL,
    SRK_btl INT NOT NULL,
    SRK_eft INT NOT NULL,
    
    -- chaves estrangeiras
    CONSTRAINT fk_pokemon
        FOREIGN KEY (SRK_pkn)
        REFERENCES dw.Dim_pkm (SRK_pkn),
        
    CONSTRAINT fk_batalha
        FOREIGN KEY (SRK_btl)
        REFERENCES dw.Dim_btl (SRK_btl),
        
    CONSTRAINT fk_efetividade
        FOREIGN KEY (SRK_eft)
        REFERENCES dw.Dim_efetContr (SRK_eft)
);
""")

conexao.commit()

Inserindo dados no banco 

In [19]:
# Usa a mesma conexao e cursor da célula anterior

# 1) Inserção na Dim_pkm
for _, row in df_tratado.iterrows():
    cursor.execute("""
        INSERT INTO dw.Dim_pkm (
            pkm_nam, tp1, tp2, hgt, wgt, gen, leg, mga_evl, all_frm, glr_frm, swt_frm
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        RETURNING SRK_pkn
    """, (
        row["pokemon_name"], row["type_1"], row["type_2"],
        row["height"], row["weight"], row["generation"],
        row["legendary"], row["mega_evolution"], row["alolan_form"],
        row["galarian_form"], row["forms_switchable"]
    ))
    srk_pkn = cursor.fetchone()[0]

    # 2) Inserção na Dim_btl
    cursor.execute("""
        INSERT INTO dw.Dim_btl (atk, dfs, cap_rte, bas_exp, exp_tpe)
        VALUES (%s, %s, %s, %s, %s)
        RETURNING SRK_btl
    """, (
        row["attack"], row["defense"], row["capture_rate"],
        row["base_experience"], row["exp_type"]
    ))
    srk_btl = cursor.fetchone()[0]

    # 3) Inserção na Dim_efetContr
    cursor.execute("""
        INSERT INTO dw.Dim_efetContr (
            agt_nrm, agt_fre, agt_wtr, agt_elt, agt_grs, agt_ice, agt_fgt, agt_psn,
            agt_gnd, agt_fly, agt_psy, agt_bug, agt_rck, agt_gst, agt_drg, agt_drk,
            agt_stl, agt_fry
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        RETURNING SRK_eft
    """, (
        row["against_normal"], row["against_fire"], row["against_water"],
        row["against_electric"], row["against_grass"], row["against_ice"],
        row["against_fighting"], row["against_poison"], row["against_ground"],
        row["against_flying"], row["against_psychic"], row["against_bug"],
        row["against_rock"], row["against_ghost"], row["against_dragon"],
        row["against_dark"], row["against_steel"], row["against_fairy"]
    ))
    srk_eft = cursor.fetchone()[0]

    # 4) Inserção na Tabela Fato (ligando as 3 dimensões)
    cursor.execute("""
        INSERT INTO dw.Fat_pokdx (SRK_pkn, SRK_btl, SRK_eft)
        VALUES (%s, %s, %s)
    """, (srk_pkn, srk_btl, srk_eft))

# Commit final
conexao.commit()


Fechando a conexão com o banco!

In [20]:
conexao.close()

print("\n Load concluído!")


 Load concluído!
